In [1]:
!oci --version

3.63.3


## Configuración de la autenticación del SDK de OCI

In [1]:
# crea carpeta y permisos
!mkdir -p /home/datascience/.oci

In [ ]:
# Ver tu HOME y listar (incluye ocultos)
!echo $HOME
!ls -la $HOME | head -n 30

In [2]:
!mkdir -p ~/.oci
!ls -la ~/.oci

total 8
drwxr-xr-x.  2 datascience users 4096 Nov  4 16:31 .
drwxr-xr-x. 14 datascience users 4096 Nov  4 16:32 ..


In [3]:
%%bash
cat > ~/.oci/config <<'CFG'
[DEFAULT]
user=ocid1.user.oc1..aaaaaaaahqepch4vmzzx3bxumt4mftlnnuoae2rnowlj3gsdsr5qt6fpsidq
tenancy=ocid1.tenancy.oc1..aaaaaaaaoi6b5sxlv4z773boczybqz3h2vspvvru42jysvizl77lky22ijaq
region=us-chicago-1
fingerprint=6d:a2:b6:25:24:40:11:8e:81:ef:df:6e:04:ce:76:9b
key_file=/home/datascience/.oci/oci_api_key.pem
CFG

echo "Config creado en ~/.oci/config"
cat ~/.oci/config | sed 's/fingerprint=.*/fingerprint=<oculto>/'

Config creado en ~/.oci/config
[DEFAULT]
user=ocid1.user.oc1..aaaaaaaahqepch4vmzzx3bxumt4mftlnnuoae2rnowlj3gsdsr5qt6fpsidq
tenancy=ocid1.tenancy.oc1..aaaaaaaaoi6b5sxlv4z773boczybqz3h2vspvvru42jysvizl77lky22ijaq
region=us-chicago-1
fingerprint=<oculto>
key_file=/home/datascience/.oci/oci_api_key.pem


In [4]:
# Quita posibles finales de línea de Windows (CRLF)
!sed -i 's/\r$//' ~/.oci/config

# mostrar
!sed -n '1,200p' ~/.oci/config

[DEFAULT]
user=ocid1.user.oc1..aaaaaaaahqepch4vmzzx3bxumt4mftlnnuoae2rnowlj3gsdsr5qt6fpsidq
tenancy=ocid1.tenancy.oc1..aaaaaaaaoi6b5sxlv4z773boczybqz3h2vspvvru42jysvizl77lky22ijaq
region=us-chicago-1
fingerprint=6d:a2:b6:25:24:40:11:8e:81:ef:df:6e:04:ce:76:9b
key_file=/home/datascience/.oci/oci_api_key.pem


# Usando los Modelos de LLM usando Python

In [5]:
user_input = "qué es llama?"

In [6]:
import oci
import json
from oci.auth.signers import get_resource_principals_signer

In [7]:
# === Config ===
REGION = "us-chicago-1"
SERVICE_ENDPOINT = f"https://inference.generativeai.{REGION}.oci.oraclecloud.com"
COMPARTMENT_ID = "ocid1.compartment.oc1..aaaaaaaaivimfwmvo5cbw6tytntxihau442eji73hqy3wsoqgewb5sjbn7ma"
MODEL_ID = "ocid1.generativeaimodel.oc1.us-chicago-1.amaaaaaask7dceya6dvgvvj3ovy4lerdl6fvx525x3yweacnrgn4ryfwwcoq"

In [8]:

# === Signer===
signer = get_resource_principals_signer()
cfg = {"region": REGION}

# (opcional)
if MODEL_ID is None:
    from oci.generative_ai import GenerativeAiClient
    genai = GenerativeAiClient(config=cfg, signer=signer)
    models = genai.list_models(
        compartment_id=COMPARTMENT_ID,
        capability=["CHAT"],
        lifecycle_state="ACTIVE"
    ).data.items
    assert models, "No hay modelos CHAT visibles en el compartimento. Revisa permisos/compartimento."
    MODEL_ID = models[0].id
    print("Usando modelo:", MODEL_ID)

# === Cliente de inferencia ===
inf = oci.generative_ai_inference.GenerativeAiInferenceClient(
    config=cfg, signer=signer, service_endpoint=SERVICE_ENDPOINT
)

# === Prompt del usuario ===
user_input = user_input

# --- Construcción del request ---
content = oci.generative_ai_inference.models.TextContent(text=user_input)
message = oci.generative_ai_inference.models.Message(role="USER", content=[content])

chat_request = oci.generative_ai_inference.models.GenericChatRequest(
    api_format=oci.generative_ai_inference.models.BaseChatRequest.API_FORMAT_GENERIC,
    messages=[message],
    max_tokens=600,
    temperature=1.0,
    frequency_penalty=0.0,
    presence_penalty=0.0,
    top_p=0.75,
)

chat_detail = oci.generative_ai_inference.models.ChatDetails(
    serving_mode=oci.generative_ai_inference.models.OnDemandServingMode(model_id=MODEL_ID),
    chat_request=chat_request,
    compartment_id=COMPARTMENT_ID,
)



In [9]:
# === Llamada ===
resp = inf.chat(chat_detail)

# === Resultado ===
choices = resp.data.chat_response.choices
response_text = choices[0].message.content[0].text if choices else "No se generó respuesta."
print(json.dumps({"response": response_text}, indent=2, ensure_ascii=False))

{
  "response": "\"Llama\" puede tener diferentes significados dependiendo del contexto. Aquí te explico los más comunes:\n\n1. **Animal**: La llama es un mamífero sudamericano de la familia de los camélidos, originario de los Andes, especialmente en países como Perú, Bolivia, Chile y Argentina. Es un animal domesticado, utilizado principalmente como animal de carga y por su lana. Es pariente de la alpaca, el guanaco y la vicuña.\n\n2. **Fuego**: En español, \"llama\" también significa \"flama\" o la parte visible del fuego. Por ejemplo: \"Las llamas del incendio eran visibles desde lejos\".\n\n3. **LLaMA (Modelo de IA)**: Si te refieres a un contexto tecnológico, LLaMA es el nombre de un modelo de inteligencia artificial desarrollado por Meta (la empresa detrás de Facebook). Significa \"Large Language Model Meta AI\" y se utiliza para tareas de procesamiento de lenguaje natural, similar a otros modelos como ChatGPT.\n\n4. **Nombre o apodo**: En algunos contextos, \"Llama\" puede ser u